# SVM–neural fusion with validation-derived uncertainty gating
Tests the next **exploratory** fusion experiment using the outputs already present on JHub.

It does **not retrain** the neural network or the RBF-SVM.

## Rationale
The downloaded RBF-SVM bundle contains **test predictions**, but not SVM predictions on the outer validation sets. Therefore, with the files currently available, test-set fusion weights are not searched.

The analysis evaluates two fusion ideas that can be defined without looking at test labels:

1. **Pre-specified equal probability fusion**
   - `0.5 × SVM + 0.5 × neural`
   - tested for both the medium Hybrid output and the medium TMC output.

2. **Validation-derived uncertainty fallback**
   - use only the neural model's **outer-validation predictions** to learn an uncertainty threshold that detects neural errors;
   - on the corresponding outer test fold:
     - use the neural probability when neural uncertainty is below the locked threshold;
     - use the SVM probability when neural uncertainty is above the locked threshold.
   - the threshold is chosen separately inside each outer fold from that fold's validation set only.

The primary comparison uses the **three common seeds 17, 42, 73** for both SVM and neural models.

## Important interpretation

The base-model test results and participant-disagreement results have already been inspected. Therefore this is a **post-hoc exploratory fusion analysis**, not a pristine model-selection stage. It is suitable for deciding what deserves external validation (e.g. AIBL), but it should not be presented as an untouched confirmatory test.

A fully validation-selected continuous fusion weight `alpha` would require SVM **validation-set probabilities**, which are not in the downloaded prediction bundle. The analysis deliberately does not manufacture or tune such a weight from test data.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

# JHub paths

RUNS_ROOT = Path("/home/jovyan/thesis_runs")

SVM_DIR = RUNS_ROOT / "svm_rbf_predictions"

NEURAL_ROOT = (
    RUNS_ROOT
    / "gated_cmt_fixed_equal_fusion_medium_v1_md050"
)

OUTPUT_ROOT = RUNS_ROOT / "svm_neural_fusion_validation_gated_v1"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [17, 42, 73]
FOLDS = [0, 1, 2, 3, 4]
THRESHOLD = 0.50

N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 20261004

print("SVM predictions:", SVM_DIR)
print("Neural experiment:", NEURAL_ROOT)
print("Output:", OUTPUT_ROOT)
print("Common seeds:", SEEDS)
print("Folds:", FOLDS)


## 1. Helper functions

The neural worker already saved `best_validation_predictions.csv` whenever a new best validation ROC AUC checkpoint was found, so the analysis can use the neural validation predictions without rerunning inference.

The SVM files available now are the saved fold-level test probabilities.


In [ ]:
def require_columns(frame, columns, context="dataframe"):
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise KeyError(
            f"{context}: missing columns {missing}. "
            f"Available: {frame.columns.tolist()}"
        )

def safe_auc(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    if np.unique(y_true).size < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_prob))

def safe_ap(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    if np.unique(y_true).size < 2:
        return np.nan
    return float(average_precision_score(y_true, y_prob))

def binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    pred = (y_prob >= threshold).astype(int)

    tp = int(np.sum((y_true == 1) & (pred == 1)))
    tn = int(np.sum((y_true == 0) & (pred == 0)))
    fp = int(np.sum((y_true == 0) & (pred == 1)))
    fn = int(np.sum((y_true == 1) & (pred == 0)))

    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    balanced_accuracy = np.nanmean([sensitivity, specificity])
    brier = float(np.mean((y_prob - y_true) ** 2))

    return {
        "roc_auc": safe_auc(y_true, y_prob),
        "average_precision": safe_ap(y_true, y_prob),
        "balanced_accuracy": float(balanced_accuracy),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "brier": brier,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

def bootstrap_auc_difference(
    y_true,
    prob_a,
    prob_b,
    n_bootstrap=2000,
    random_seed=20261004,
):
    """Stratified participant bootstrap for AUC(A)-AUC(B)."""
    y_true = np.asarray(y_true, dtype=int)
    prob_a = np.asarray(prob_a, dtype=float)
    prob_b = np.asarray(prob_b, dtype=float)

    idx0 = np.flatnonzero(y_true == 0)
    idx1 = np.flatnonzero(y_true == 1)

    rng = np.random.default_rng(random_seed)
    diffs = np.empty(n_bootstrap, dtype=float)

    for b in range(n_bootstrap):
        sample0 = rng.choice(idx0, size=len(idx0), replace=True)
        sample1 = rng.choice(idx1, size=len(idx1), replace=True)
        sample = np.concatenate([sample0, sample1])

        diffs[b] = (
            roc_auc_score(y_true[sample], prob_a[sample])
            - roc_auc_score(y_true[sample], prob_b[sample])
        )

    observed = safe_auc(y_true, prob_a) - safe_auc(y_true, prob_b)

    return {
        "auc_difference": float(observed),
        "ci_low": float(np.quantile(diffs, 0.025)),
        "ci_high": float(np.quantile(diffs, 0.975)),
        "bootstrap_sd": float(np.std(diffs, ddof=1)),
        "prob_difference_gt_0": float(np.mean(diffs > 0)),
    }

def error_detection_metrics(error_true, score, threshold):
    """Treat neural error as the positive class and uncertainty as the score."""
    error_true = np.asarray(error_true, dtype=int)
    score = np.asarray(score, dtype=float)
    pred_error = (score >= threshold).astype(int)

    tp = int(np.sum((error_true == 1) & (pred_error == 1)))
    tn = int(np.sum((error_true == 0) & (pred_error == 0)))
    fp = int(np.sum((error_true == 0) & (pred_error == 1)))
    fn = int(np.sum((error_true == 1) & (pred_error == 0)))

    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    balanced_accuracy = np.nanmean([sensitivity, specificity])

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = sensitivity
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else 0.0
    )

    return {
        "threshold": float(threshold),
        "balanced_accuracy": float(balanced_accuracy),
        "sensitivity_to_errors": float(sensitivity),
        "specificity_to_correct": float(specificity),
        "precision_for_errors": float(precision),
        "f1_for_errors": float(f1),
        "flagged_fraction": float(pred_error.mean()),
    }

def choose_uncertainty_threshold(validation_table, prob_col, uncertainty_col):
    """Choose threshold using validation neural-error detection only."""
    require_columns(
        validation_table,
        ["TARGET", prob_col, uncertainty_col],
        "validation table",
    )

    y = validation_table["TARGET"].to_numpy(dtype=int)
    p = validation_table[prob_col].to_numpy(dtype=float)
    u = validation_table[uncertainty_col].to_numpy(dtype=float)

    neural_error = ((p >= THRESHOLD).astype(int) != y).astype(int)

    if neural_error.sum() == 0 or neural_error.sum() == len(neural_error):
        # Degenerate case: use the 75th percentile as a deterministic fallback.
        threshold = float(np.quantile(u, 0.75))
        metrics = error_detection_metrics(neural_error, u, threshold)
        metrics["selection_note"] = "degenerate_error_labels_q75_fallback"
        return threshold, pd.DataFrame([metrics])

    # Candidate thresholds from observed validation uncertainty values.
    candidates = np.unique(
        np.quantile(
            u,
            np.linspace(0.05, 0.95, 91),
        )
    )

    rows = [
        error_detection_metrics(neural_error, u, t)
        for t in candidates
    ]

    ranking = pd.DataFrame(rows).sort_values(
        [
            "balanced_accuracy",
            "f1_for_errors",
            "specificity_to_correct",
        ],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    selected = float(ranking.iloc[0]["threshold"])
    ranking["selection_note"] = "validation_error_detection"

    return selected, ranking

def load_neural_seed_file(fold, seed, split):
    if split == "validation":
        filename = "best_validation_predictions.csv"
    elif split == "test":
        filename = "test_predictions.csv"
    else:
        raise ValueError(split)

    path = (
        NEURAL_ROOT
        / f"seed_{seed}"
        / "mci_prognosis"
        / f"fold_{fold}"
        / "predictions"
        / filename
    )

    if not path.exists():
        raise FileNotFoundError(path)

    frame = pd.read_csv(path)

    require_columns(
        frame,
        [
            "RID",
            "TARGET",
            "FINAL_P_pMCI",
            "FINAL_UNCERTAINTY",
            "TMC_P_pMCI",
            "TMC_UNCERTAINTY",
        ],
        str(path),
    )

    frame = frame.copy()
    frame["SEED"] = seed
    frame["FOLD"] = fold
    return frame

def aggregate_neural_split(fold, split):
    frames = [
        load_neural_seed_file(fold, seed, split)
        for seed in SEEDS
    ]

    raw = pd.concat(frames, ignore_index=True)

    # Targets and participant sets must be identical across seeds.
    counts = raw.groupby("RID")["SEED"].nunique()
    if not (counts == len(SEEDS)).all():
        raise ValueError(
            f"Fold {fold} {split}: not every RID has all {len(SEEDS)} seeds."
        )

    target_nunique = raw.groupby("RID")["TARGET"].nunique()
    if not (target_nunique == 1).all():
        raise ValueError(
            f"Fold {fold} {split}: target mismatch across neural seeds."
        )

    out = (
        raw.groupby("RID")
        .agg(
            TARGET=("TARGET", "first"),
            HYBRID_P=("FINAL_P_pMCI", "mean"),
            HYBRID_P_SD=("FINAL_P_pMCI", "std"),
            HYBRID_U=("FINAL_UNCERTAINTY", "mean"),
            TMC_P=("TMC_P_pMCI", "mean"),
            TMC_P_SD=("TMC_P_pMCI", "std"),
            TMC_U=("TMC_UNCERTAINTY", "mean"),
        )
        .reset_index()
    )

    out["FOLD"] = fold
    out["SPLIT"] = split
    return out

def load_svm_test_fold(fold):
    frames = []

    for seed in SEEDS:
        path = (
            SVM_DIR
            / f"svm_rbf_fold_{fold}_seed_{seed}_test_predictions.csv"
        )

        if not path.exists():
            raise FileNotFoundError(path)

        frame = pd.read_csv(path)
        require_columns(
            frame,
            ["RID", "TARGET", "PROB_PMCI"],
            str(path),
        )

        frame = frame.copy()
        frame["SEED"] = seed
        frames.append(frame)

    raw = pd.concat(frames, ignore_index=True)

    counts = raw.groupby("RID")["SEED"].nunique()
    if not (counts == len(SEEDS)).all():
        raise ValueError(
            f"Fold {fold}: not every SVM RID has all {len(SEEDS)} seeds."
        )

    target_nunique = raw.groupby("RID")["TARGET"].nunique()
    if not (target_nunique == 1).all():
        raise ValueError(
            f"Fold {fold}: SVM target mismatch across seeds."
        )

    out = (
        raw.groupby("RID")
        .agg(
            TARGET=("TARGET", "first"),
            SVM_P=("PROB_PMCI", "mean"),
            SVM_P_SD=("PROB_PMCI", "std"),
        )
        .reset_index()
    )

    out["FOLD"] = fold
    return out


## 2. Input checks

This confirms that all neural validation/test files and all 15 SVM test files exist before any fusion result is calculated.


In [ ]:
missing = []

for fold in FOLDS:
    for seed in SEEDS:
        for filename in [
            "best_validation_predictions.csv",
            "test_predictions.csv",
        ]:
            path = (
                NEURAL_ROOT
                / f"seed_{seed}"
                / "mci_prognosis"
                / f"fold_{fold}"
                / "predictions"
                / filename
            )
            if not path.exists():
                missing.append(str(path))

        svm_path = (
            SVM_DIR
            / f"svm_rbf_fold_{fold}_seed_{seed}_test_predictions.csv"
        )
        if not svm_path.exists():
            missing.append(str(svm_path))

if missing:
    print("Missing required files:")
    for path in missing:
        print(" -", path)
    raise FileNotFoundError(
        f"{len(missing)} required file(s) are missing."
    )

print("Input check passed.")
print("All medium neural validation/test predictions are present.")
print("All 15 SVM fold-level test prediction files are present.")


## 3. Build validation-locked fusion rules fold by fold

For each outer fold:

- average the three neural seeds on the **validation set**;
- select an uncertainty threshold for detecting Hybrid errors;
- select a separate uncertainty threshold for detecting TMC errors;
- lock those thresholds;
- never use the outer-test labels during threshold selection.

The selected thresholds and validation error-detection performance are saved for auditability.


In [ ]:
fold_rules = []
validation_rankings = {}

for fold in FOLDS:
    val = aggregate_neural_split(fold, "validation")

    hybrid_threshold, hybrid_ranking = choose_uncertainty_threshold(
        val,
        prob_col="HYBRID_P",
        uncertainty_col="HYBRID_U",
    )

    tmc_threshold, tmc_ranking = choose_uncertainty_threshold(
        val,
        prob_col="TMC_P",
        uncertainty_col="TMC_U",
    )

    validation_rankings[(fold, "hybrid")] = hybrid_ranking
    validation_rankings[(fold, "tmc")] = tmc_ranking

    hybrid_best = hybrid_ranking.iloc[0]
    tmc_best = tmc_ranking.iloc[0]

    fold_rules.append({
        "FOLD": fold,
        "N_VALIDATION": len(val),

        "HYBRID_VALIDATION_AUC":
            safe_auc(val["TARGET"], val["HYBRID_P"]),

        "TMC_VALIDATION_AUC":
            safe_auc(val["TARGET"], val["TMC_P"]),

        "HYBRID_U_THRESHOLD":
            hybrid_threshold,

        "HYBRID_ERROR_DETECTION_BAL_ACC":
            hybrid_best["balanced_accuracy"],

        "HYBRID_ERROR_DETECTION_SENSITIVITY":
            hybrid_best["sensitivity_to_errors"],

        "HYBRID_FLAGGED_FRACTION":
            hybrid_best["flagged_fraction"],

        "TMC_U_THRESHOLD":
            tmc_threshold,

        "TMC_ERROR_DETECTION_BAL_ACC":
            tmc_best["balanced_accuracy"],

        "TMC_ERROR_DETECTION_SENSITIVITY":
            tmc_best["sensitivity_to_errors"],

        "TMC_FLAGGED_FRACTION":
            tmc_best["flagged_fraction"],
    })

    hybrid_ranking.to_csv(
        TABLES_DIR
        / f"fold_{fold}_hybrid_uncertainty_threshold_ranking.csv",
        index=False,
    )

    tmc_ranking.to_csv(
        TABLES_DIR
        / f"fold_{fold}_tmc_uncertainty_threshold_ranking.csv",
        index=False,
    )

fold_rules = pd.DataFrame(fold_rules)

display(fold_rules)

fold_rules.to_csv(
    TABLES_DIR / "validation_locked_uncertainty_rules.csv",
    index=False,
)


## 4. Apply the locked rules to each outer test fold

Methods evaluated:

- `svm`
- `hybrid_neural`
- `tmc_neural`
- `equal_svm_hybrid` = 0.5 SVM + 0.5 Hybrid
- `equal_svm_tmc` = 0.5 SVM + 0.5 TMC
- `uncertainty_fallback_hybrid`
  - Hybrid when validation-derived Hybrid uncertainty rule says "trust neural"
  - SVM when the rule says "fallback"
- `uncertainty_fallback_tmc`
  - same logic using TMC uncertainty

No test-set threshold or weight is optimized.


In [ ]:
test_fold_tables = []
fold_metric_rows = []

for fold in FOLDS:
    neural_test = aggregate_neural_split(fold, "test")
    svm_test = load_svm_test_fold(fold)

    merged = neural_test.merge(
        svm_test[
            ["RID", "TARGET", "SVM_P", "SVM_P_SD"]
        ],
        on="RID",
        how="inner",
        suffixes=("_NEURAL", "_SVM"),
        validate="one_to_one",
    )

    if len(merged) != len(neural_test):
        raise ValueError(
            f"Fold {fold}: participant mismatch between neural and SVM test predictions."
        )

    if not (
        merged["TARGET_NEURAL"].astype(int)
        == merged["TARGET_SVM"].astype(int)
    ).all():
        raise ValueError(
            f"Fold {fold}: neural and SVM test targets do not match."
        )

    merged = merged.rename(
        columns={"TARGET_NEURAL": "TARGET"}
    ).drop(columns=["TARGET_SVM"])

    rule = fold_rules.loc[
        fold_rules["FOLD"] == fold
    ].iloc[0]

    # Pre-specified equal fusion

    merged["P_SVM"] = merged["SVM_P"]
    merged["P_HYBRID"] = merged["HYBRID_P"]
    merged["P_TMC"] = merged["TMC_P"]

    merged["P_EQUAL_SVM_HYBRID"] = (
        0.5 * merged["SVM_P"]
        + 0.5 * merged["HYBRID_P"]
    )

    merged["P_EQUAL_SVM_TMC"] = (
        0.5 * merged["SVM_P"]
        + 0.5 * merged["TMC_P"]
    )

    # Validation-derived hard fallback

    merged["HYBRID_FALLBACK_TO_SVM"] = (
        merged["HYBRID_U"]
        >= float(rule["HYBRID_U_THRESHOLD"])
    ).astype(int)

    merged["TMC_FALLBACK_TO_SVM"] = (
        merged["TMC_U"]
        >= float(rule["TMC_U_THRESHOLD"])
    ).astype(int)

    merged["P_UNCERTAINTY_FALLBACK_HYBRID"] = np.where(
        merged["HYBRID_FALLBACK_TO_SVM"] == 1,
        merged["SVM_P"],
        merged["HYBRID_P"],
    )

    merged["P_UNCERTAINTY_FALLBACK_TMC"] = np.where(
        merged["TMC_FALLBACK_TO_SVM"] == 1,
        merged["SVM_P"],
        merged["TMC_P"],
    )

    merged["FOLD"] = fold

    test_fold_tables.append(merged)

    method_columns = {
        "svm": "P_SVM",
        "hybrid_neural": "P_HYBRID",
        "tmc_neural": "P_TMC",
        "equal_svm_hybrid": "P_EQUAL_SVM_HYBRID",
        "equal_svm_tmc": "P_EQUAL_SVM_TMC",
        "uncertainty_fallback_hybrid":
            "P_UNCERTAINTY_FALLBACK_HYBRID",
        "uncertainty_fallback_tmc":
            "P_UNCERTAINTY_FALLBACK_TMC",
    }

    for method, col in method_columns.items():
        metrics = binary_metrics(
            merged["TARGET"],
            merged[col],
        )

        row = {
            "FOLD": fold,
            "METHOD": method,
            "N": len(merged),
            **metrics,
        }

        if method == "uncertainty_fallback_hybrid":
            row["FALLBACK_FRACTION"] = float(
                merged["HYBRID_FALLBACK_TO_SVM"].mean()
            )

        elif method == "uncertainty_fallback_tmc":
            row["FALLBACK_FRACTION"] = float(
                merged["TMC_FALLBACK_TO_SVM"].mean()
            )

        else:
            row["FALLBACK_FRACTION"] = np.nan

        fold_metric_rows.append(row)

test_predictions = pd.concat(
    test_fold_tables,
    ignore_index=True,
)

fold_metrics = pd.DataFrame(fold_metric_rows)

display(fold_metrics)

test_predictions.to_csv(
    TABLES_DIR / "all_test_participant_fusion_predictions.csv",
    index=False,
)

fold_metrics.to_csv(
    TABLES_DIR / "fusion_metrics_by_fold.csv",
    index=False,
)


## 5. Pooled out-of-fold results across all 544 participants

This is the main descriptive comparison.


In [ ]:
method_columns = {
    "svm": "P_SVM",
    "hybrid_neural": "P_HYBRID",
    "tmc_neural": "P_TMC",
    "equal_svm_hybrid": "P_EQUAL_SVM_HYBRID",
    "equal_svm_tmc": "P_EQUAL_SVM_TMC",
    "uncertainty_fallback_hybrid":
        "P_UNCERTAINTY_FALLBACK_HYBRID",
    "uncertainty_fallback_tmc":
        "P_UNCERTAINTY_FALLBACK_TMC",
}

pooled_rows = []

for method, col in method_columns.items():
    metrics = binary_metrics(
        test_predictions["TARGET"],
        test_predictions[col],
    )

    pooled_rows.append({
        "METHOD": method,
        "N": len(test_predictions),
        **metrics,
    })

pooled_metrics = (
    pd.DataFrame(pooled_rows)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

display(pooled_metrics)

pooled_metrics.to_csv(
    TABLES_DIR / "pooled_fusion_metrics.csv",
    index=False,
)

print("\nPooled ROC AUC ranking:")
for _, row in pooled_metrics.iterrows():
    print(
        f"{row['METHOD']:<32s} "
        f"AUC={row['roc_auc']:.6f}  "
        f"AP={row['average_precision']:.6f}  "
        f"Brier={row['brier']:.6f}"
    )


## 6. Paired bootstrap: does a fusion improve on SVM or the neural model?

Because all methods are evaluated on exactly the same participants, paired participant bootstrap is more informative than comparing point estimates alone.

The notebook reports the 95% bootstrap interval for the AUC difference.


In [ ]:
comparisons = [
    ("equal_svm_hybrid", "svm"),
    ("equal_svm_hybrid", "hybrid_neural"),
    ("equal_svm_tmc", "svm"),
    ("equal_svm_tmc", "tmc_neural"),
    ("uncertainty_fallback_hybrid", "svm"),
    ("uncertainty_fallback_hybrid", "hybrid_neural"),
    ("uncertainty_fallback_tmc", "svm"),
    ("uncertainty_fallback_tmc", "tmc_neural"),
]

bootstrap_rows = []

for i, (method_a, method_b) in enumerate(comparisons):
    col_a = method_columns[method_a]
    col_b = method_columns[method_b]

    result = bootstrap_auc_difference(
        test_predictions["TARGET"],
        test_predictions[col_a],
        test_predictions[col_b],
        n_bootstrap=N_BOOTSTRAP,
        random_seed=BOOTSTRAP_SEED + i,
    )

    bootstrap_rows.append({
        "METHOD_A": method_a,
        "METHOD_B": method_b,
        **result,
    })

bootstrap_comparisons = pd.DataFrame(bootstrap_rows)

display(bootstrap_comparisons)

bootstrap_comparisons.to_csv(
    TABLES_DIR / "paired_bootstrap_auc_differences.csv",
    index=False,
)


## 7. Who is actually rescued by the fusion?

This counts threshold-level rescue at 0.5. It is separate from ROC AUC, but useful for understanding whether the fusion is solving the disagreement cases that motivated it.


In [ ]:
def correctness(y, p):
    return ((np.asarray(p) >= 0.5).astype(int) == np.asarray(y)).astype(int)

y = test_predictions["TARGET"].to_numpy(dtype=int)

test_predictions["CORRECT_SVM"] = correctness(
    y,
    test_predictions["P_SVM"],
)

test_predictions["CORRECT_HYBRID"] = correctness(
    y,
    test_predictions["P_HYBRID"],
)

test_predictions["CORRECT_EQUAL_SVM_HYBRID"] = correctness(
    y,
    test_predictions["P_EQUAL_SVM_HYBRID"],
)

test_predictions["CORRECT_U_FALLBACK_HYBRID"] = correctness(
    y,
    test_predictions["P_UNCERTAINTY_FALLBACK_HYBRID"],
)

rescue_rows = []

for method, correct_col in [
    ("equal_svm_hybrid", "CORRECT_EQUAL_SVM_HYBRID"),
    ("uncertainty_fallback_hybrid", "CORRECT_U_FALLBACK_HYBRID"),
]:
    fusion_correct = test_predictions[correct_col]

    rescue_rows.append({
        "METHOD": method,

        "NEURAL_WRONG_FUSION_RIGHT": int(
            (
                (test_predictions["CORRECT_HYBRID"] == 0)
                & (fusion_correct == 1)
            ).sum()
        ),

        "NEURAL_RIGHT_FUSION_WRONG": int(
            (
                (test_predictions["CORRECT_HYBRID"] == 1)
                & (fusion_correct == 0)
            ).sum()
        ),

        "SVM_WRONG_FUSION_RIGHT": int(
            (
                (test_predictions["CORRECT_SVM"] == 0)
                & (fusion_correct == 1)
            ).sum()
        ),

        "SVM_RIGHT_FUSION_WRONG": int(
            (
                (test_predictions["CORRECT_SVM"] == 1)
                & (fusion_correct == 0)
            ).sum()
        ),
    })

rescue_summary = pd.DataFrame(rescue_rows)

display(rescue_summary)

rescue_summary.to_csv(
    TABLES_DIR / "fusion_rescue_counts.csv",
    index=False,
)


## 8. Fold-by-fold visualization

A fusion should ideally improve consistently rather than depending on a single unusually favorable fold.


In [ ]:
plot_methods = [
    "svm",
    "hybrid_neural",
    "equal_svm_hybrid",
    "uncertainty_fallback_hybrid",
]

plot_df = fold_metrics.loc[
    fold_metrics["METHOD"].isin(plot_methods)
].copy()

fig, ax = plt.subplots(figsize=(9, 5))

for method in plot_methods:
    g = (
        plot_df.loc[plot_df["METHOD"] == method]
        .sort_values("FOLD")
    )

    ax.plot(
        g["FOLD"],
        g["roc_auc"],
        marker="o",
        label=method,
    )

ax.set_xlabel("Outer fold")
ax.set_ylabel("ROC AUC")
ax.set_title("SVM–Hybrid fusion performance by outer fold")
ax.set_xticks(FOLDS)
ax.legend()
fig.tight_layout()

path = FIGURES_DIR / "fusion_auc_by_fold.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", path)


## 9. Summary

Compare the pooled score with fold consistency and paired uncertainty. Look for:

- whether equal fusion beats the SVM at all;
- whether the uncertainty fallback beats the SVM;
- whether any gain is reasonably consistent across folds;
- whether the paired-bootstrap AUC interval supports a real improvement rather than noise;
- whether the uncertainty fallback actually rescues neural errors without introducing too many new errors.

If neither fusion improves on the SVM, that is still a useful result: the models have complementary threshold errors, but their probability rankings may not combine in a way that improves ROC AUC.


In [ ]:
print("=" * 88)
print("SVM–NEURAL FUSION SUMMARY")
print("=" * 88)

print("\nValidation-locked uncertainty rules:")
display(
    fold_rules[
        [
            "FOLD",
            "HYBRID_U_THRESHOLD",
            "HYBRID_ERROR_DETECTION_BAL_ACC",
            "HYBRID_FLAGGED_FRACTION",
            "TMC_U_THRESHOLD",
            "TMC_ERROR_DETECTION_BAL_ACC",
            "TMC_FLAGGED_FRACTION",
        ]
    ]
)

print("\nPooled performance:")
display(
    pooled_metrics[
        [
            "METHOD",
            "roc_auc",
            "average_precision",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "brier",
        ]
    ]
)

print("\nPaired bootstrap AUC differences:")
display(
    bootstrap_comparisons[
        [
            "METHOD_A",
            "METHOD_B",
            "auc_difference",
            "ci_low",
            "ci_high",
            "prob_difference_gt_0",
        ]
    ]
)

print("\nThreshold-level rescue counts:")
display(rescue_summary)

print("\nSaved tables:")
for p in sorted(TABLES_DIR.glob("*.csv")):
    print(" -", p)

print("\nSaved figures:")
for p in sorted(FIGURES_DIR.glob("*.png")):
    print(" -", p)

print(
    "\nMethodological note: no test-set fusion weight was tuned. "
    "Equal fusion was fixed at 0.5/0.5, and uncertainty fallback thresholds "
    "were selected only from each outer fold's neural validation predictions."
)


## Optional next extension: validation-selected continuous alpha

A continuous fusion

\[
p = \alpha p_{SVM} + (1-\alpha)p_{neural}
\]

should only be added once we have **SVM probabilities for each outer validation set**. Then `alpha` can be selected on validation and locked before test evaluation.

Do **not** scan `alpha` on the test predictions in this notebook.
